# Preregistered Qwen pilot v0.8

Select a GPU runtime and choose **Run all**. This notebook attempts seeds
401 to 405 in order using upper-layer training and extra neighbor rehearsal.
The first qualified seed receives the three-candidate smoke run, followed
by the full 19-candidate sweep if the smoke checks pass.

[Public preregistration](https://github.com/AjayKasu1/intervention-ladder/blob/e3831c15f42169ef03b65ecbe3c61157d9fadc17/docs/PILOT_V08.md).
Keep the runtime connected until the reports download.


In [ ]:
# @title 1. Load the public preregistration revision
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

CODE_e3831c15f42169ef03b65ecbe3c61157d9fadc17 = "e3831c15f42169ef03b65ecbe3c61157d9fadc17"
REPO = Path("/content/intervention-ladder-pilot-v08")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU runtime before running the pilot.")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AjayKasu1/intervention-ladder.git", str(REPO)], check=True)
subprocess.run(["git", "merge-base", "--is-ancestor", CODE_e3831c15f42169ef03b65ecbe3c61157d9fadc17, "origin/main"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_e3831c15f42169ef03b65ecbe3c61157d9fadc17], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.57.6",
                "numpy>=1.26", "scipy>=1.11", "pyyaml>=6.0"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access CUDA.")
manifest = {
    "run_mode": "pilot", "pilot_version": "0.8", "code_revision": CODE_e3831c15f42169ef03b65ecbe3c61157d9fadc17,
    "config_sha256": hashlib.sha256((REPO / "configs/pilot_v08.yaml").read_bytes()).hexdigest(),
    "gpu": torch.cuda.get_device_name(0), "python": sys.version.split()[0],
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "numpy", "scipy", "pyyaml")},
}
print("GPU:", manifest["gpu"], "| Preregistered revision:", CODE_e3831c15f42169ef03b65ecbe3c61157d9fadc17)


In [ ]:
# @title 2. Run the fixed v0.8 cohort
WORK = REPO / "artifacts/pilot_v08"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2))
with (WORK / "pilot.log").open("w") as log, subprocess.Popen(
    [sys.executable, "-u", "scripts/run_pilot.py", "--config", "configs/pilot_v08.yaml"],
    cwd=REPO, env={**os.environ, "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"},
    text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    exit_status = process.wait()
print("Pilot process exit status:", exit_status)


In [ ]:
# @title 3. Download the cohort and sweep reports
from google.colab import files
ledger_path = WORK / "pilot.json"
if not ledger_path.exists():
    raise RuntimeError("No pilot ledger exists. Review the preceding cell.")
report = json.loads(ledger_path.read_text())
bundle = WORK / "qwen_pilot_v08_reports.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in [ledger_path, WORK / "manifest.json", WORK / "pilot.log"]:
        if item.is_file():
            archive.write(item, item.name)
    for case in report["cases"]:
        directory = REPO / "results" / case["case_id"]
        for name in ("smoke.json", "exhaustive.json"):
            item = directory / name
            if item.is_file():
                archive.write(item, case["case_id"] + "/" + name)
print("Qualification:", report["qualified"], "of", report["attempted"])
print("Cases:", [(c["seed"], c["status"]) for c in report["cases"]])
print("First qualified:", report["first_qualified"], "| Sweep:", report["sweep_status"])
files.download(str(bundle))
